# AIO Presence & Overlap Analysis
Loads all `serp_raw_*.json` files and analyses when Google AI Overviews appear and how much they overlap with organic results.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
PALETTE = {"pro": "#2ecc71", "neutrale": "#3498db", "contro": "#e74c3c"}

ROOT = Path("..").resolve()
RAW_DIR = ROOT / "data" / "raw"

records = []
for path in sorted(RAW_DIR.glob("serp_raw_*.json")):
    with open(path) as f:
        batch = json.load(f)
    for r in batch:
        r["_file"] = path.name
    records.extend(batch)

df = pd.DataFrame(records)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)
df["has_ai_overview"] = df["has_ai_overview"].astype(bool)
df["aio_organic_overlap"] = pd.to_numeric(df["aio_organic_overlap"], errors="coerce")

print(f"Total records : {len(df)}")
print(f"Topics        : {sorted(df['topic'].dropna().unique())}")
print(
    f"Date range    : {df['timestamp_utc'].min().date()} → {df['timestamp_utc'].max().date()}"
)
df.head(3)

## 1. AIO presence per topic

In [ ]:
topic_stats = (
    df.groupby("topic")["has_ai_overview"]
    .agg(total="count", aio_present="sum")
    .assign(
        aio_absent=lambda x: x["total"] - x["aio_present"],
        aio_rate=lambda x: x["aio_present"] / x["total"],
    )
    .sort_values("aio_rate", ascending=False)
)
display(topic_stats.style.format({"aio_rate": "{:.0%}"}))

fig, ax = plt.subplots(figsize=(10, 4))
x = range(len(topic_stats))
ax.bar(x, topic_stats["aio_present"], label="AIO present", color="steelblue")
ax.bar(
    x,
    topic_stats["aio_absent"],
    bottom=topic_stats["aio_present"],
    label="AIO absent",
    color="#d9534f",
)
ax.set_xticks(list(x))
ax.set_xticklabels(topic_stats.index, rotation=30, ha="right")
ax.set_ylabel("Number of requests")
ax.set_title("AIO presence per topic")
ax.legend()
plt.tight_layout()
plt.show()

## 2. AIO presence per stance (pro / neutrale / contro)

In [ ]:
df_stance = df[df["stance"].notna()]

if df_stance.empty:
    print("No stance data yet — run generate_queries.py + serpapi_collector.py first.")
else:
    stance_stats = (
        df_stance.groupby("stance")["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reindex(["pro", "neutrale", "contro"])
    )
    display(stance_stats.style.format({"aio_rate": "{:.0%}"}))

    fig, ax = plt.subplots(figsize=(6, 4))
    for i, (stance, row) in enumerate(stance_stats.iterrows()):
        ax.bar(i, row["aio_rate"], color=PALETTE.get(stance, "grey"), label=stance)
    ax.set_xticks(range(len(stance_stats)))
    ax.set_xticklabels(stance_stats.index)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO presence rate")
    ax.set_title("AIO rate by query stance")
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 3. Consistency — repeated queries
For every query run more than once, did AIO appear consistently?

In [ ]:
consistency = (
    df.groupby("query")["has_ai_overview"]
    .agg(runs="count", aio_count="sum")
    .query("runs > 1")
    .assign(
        aio_rate=lambda x: x["aio_count"] / x["runs"],
        consistent=lambda x: (x["aio_count"] == x["runs"]) | (x["aio_count"] == 0),
        verdict=lambda x: x.apply(
            lambda r: (
                "always"
                if r["aio_count"] == r["runs"]
                else ("never" if r["aio_count"] == 0 else "inconsistent")
            ),
            axis=1,
        ),
    )
    .sort_values(["verdict", "query"])
)

print(consistency["verdict"].value_counts().to_string())
display(consistency)

In [ ]:
inconsistent = consistency[consistency["verdict"] == "inconsistent"]
if inconsistent.empty:
    print("No inconsistent queries.")
else:
    print(f"{len(inconsistent)} queries with inconsistent AIO:")
    display(inconsistent[["runs", "aio_count", "aio_rate"]])
    detail = df[df["query"].isin(inconsistent.index)][
        ["query", "timestamp_utc", "has_ai_overview"]
    ].sort_values(["query", "timestamp_utc"])
    display(detail)

## 4. AIO presence per subtopic

In [ ]:
df_sub = df[df["subtopic"].notna()]

if df_sub.empty:
    print("No subtopic data yet.")
else:
    subtopic_stats = (
        df_sub.groupby(["topic", "subtopic"])["has_ai_overview"]
        .agg(total="count", aio_present="sum")
        .assign(aio_rate=lambda x: x["aio_present"] / x["total"])
        .reset_index()
        .sort_values(["topic", "aio_rate"], ascending=[True, False])
    )

    for topic, group in subtopic_stats.groupby("topic"):
        fig, ax = plt.subplots(figsize=(10, max(3, len(group) * 0.5)))
        bars = ax.barh(group["subtopic"], group["aio_rate"], color="steelblue")
        ax.bar_label(bars, fmt="{:.0%}", padding=4)
        ax.set_xlim(0, 1.15)
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_title(f"AIO rate by subtopic — {topic}")
        ax.invert_yaxis()
        plt.tight_layout()
        plt.show()

## 5. AIO–Organic Overlap
`aio_organic_overlap` = share of AIO-cited domains that also appear in the organic top-10.  
Only computed for records where AIO was present.

In [ ]:
df_aio = df[df["has_ai_overview"] & df["aio_organic_overlap"].notna()].copy()
print(f"Records with AIO + overlap score: {len(df_aio)}")
print(df_aio["aio_organic_overlap"].describe().to_string())

In [ ]:
# ── 5a. Per topic ─────────────────────────────────────────────────────────────
topics = sorted(df_aio["topic"].dropna().unique())

fig, ax = plt.subplots(figsize=(max(6, len(topics) * 1.5), 5))
sns.boxplot(
    data=df_aio,
    x="topic",
    y="aio_organic_overlap",
    order=topics,
    color="steelblue",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_aio,
    x="topic",
    y="aio_organic_overlap",
    order=topics,
    color="black",
    size=4,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics, rotation=30, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("AIO–organic overlap")
ax.set_xlabel("")
ax.set_title("AIO–organic overlap per topic")
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5b. Per subtopic (one plot per topic) ─────────────────────────────────────
df_aio_sub = df_aio[df_aio["subtopic"].notna()]

if df_aio_sub.empty:
    print("No subtopic data with overlap scores yet.")
else:
    for topic, group in df_aio_sub.groupby("topic"):
        subtopics = (
            group.groupby("subtopic")["aio_organic_overlap"]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        fig, ax = plt.subplots(figsize=(11, max(4, len(subtopics) * 0.7)))
        sns.boxplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            order=subtopics,
            color="steelblue",
            width=0.5,
            orient="h",
            ax=ax,
        )
        sns.stripplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            order=subtopics,
            color="black",
            size=4,
            alpha=0.5,
            jitter=True,
            orient="h",
            ax=ax,
        )
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_xlim(-0.05, 1.05)
        ax.set_xlabel("AIO–organic overlap")
        ax.set_ylabel("")
        ax.set_title(f"AIO–organic overlap per subtopic — {topic}")
        plt.tight_layout()
        plt.show()

In [ ]:
# ── 5c. Per stance ────────────────────────────────────────────────────────────
df_aio_stance = df_aio[df_aio["stance"].notna()]

if df_aio_stance.empty:
    print("No stance data with overlap scores yet.")
else:
    order = ["pro", "neutrale", "contro"]
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(
        data=df_aio_stance,
        x="stance",
        y="aio_organic_overlap",
        order=order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_aio_stance,
        x="stance",
        y="aio_organic_overlap",
        order=order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("AIO–organic overlap")
    ax.set_xlabel("Query stance")
    ax.set_title("AIO–organic overlap by stance")
    ax.set_ylim(-0.05, 1.05)
    plt.tight_layout()
    plt.show()

    print("\nMedian overlap per stance:")
    print(df_aio_stance.groupby("stance")["aio_organic_overlap"].describe().round(2))

In [ ]:
# ── 5d. Combined: subtopic × stance (one facet grid per topic) ────────────────
df_aio_full = df_aio[df_aio["subtopic"].notna() & df_aio["stance"].notna()]

if df_aio_full.empty:
    print("No combined subtopic+stance data yet.")
else:
    for topic, group in df_aio_full.groupby("topic"):
        subtopics = (
            group.groupby("subtopic")["aio_organic_overlap"]
            .median()
            .sort_values(ascending=False)
            .index.tolist()
        )
        fig, ax = plt.subplots(figsize=(12, max(4, len(subtopics) * 0.8)))
        sns.boxplot(
            data=group,
            y="subtopic",
            x="aio_organic_overlap",
            hue="stance",
            order=subtopics,
            hue_order=["pro", "neutrale", "contro"],
            palette=PALETTE,
            width=0.6,
            orient="h",
            ax=ax,
        )
        ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
        ax.set_xlim(-0.05, 1.05)
        ax.set_xlabel("AIO–organic overlap")
        ax.set_ylabel("")
        ax.set_title(f"AIO–organic overlap by subtopic × stance — {topic}")
        ax.legend(title="stance", bbox_to_anchor=(1.01, 1), loc="upper left")
        plt.tight_layout()
        plt.show()

## 6. UGC Sources: Organic vs AIO

How often do UGC platforms (YouTube, Reddit, Twitter/X, etc.) appear in organic top-10 results versus AIO citations?  
`ugc_share_organic` = fraction of organic top-10 domains that are UGC; `ugc_share_aio` = same for AIO sources (only for records where AIO appeared).

In [ ]:
UGC_DOMAINS = {
    "facebook.com",
    "instagram.com",
    "youtube.com",
    "reddit.com",
    "tiktok.com",
    "twitter.com",
    "x.com",
    "threads.com",
    "linkedin.com",
    "pinterest.com",
    "quora.com",
}


def ugc_share(domains_json):
    """Fraction of domains in a JSON-encoded list that belong to a UGC platform."""
    if not domains_json or (isinstance(domains_json, float)):
        return None
    try:
        domains = (
            json.loads(domains_json) if isinstance(domains_json, str) else domains_json
        )
    except Exception:
        return None
    if not domains:
        return None
    ugc = sum(
        1 for d in domains if any(d == u or d.endswith("." + u) for u in UGC_DOMAINS)
    )
    return ugc / len(domains)


df["ugc_share_organic"] = df["organic_domains"].apply(ugc_share)
df["ugc_share_aio"] = df["aio_domains"].apply(ugc_share)  # NaN where no AIO

n_aio_rows = df["has_ai_overview"].sum()
n_org_ugc = (df["ugc_share_organic"].fillna(0) > 0).sum()
n_aio_ugc = (df.loc[df["has_ai_overview"], "ugc_share_aio"].fillna(0) > 0).sum()

print(
    f"Queries with ≥1 UGC in organic  : {n_org_ugc}/{len(df)} ({n_org_ugc / len(df):.0%})"
)
print(
    f"Queries with ≥1 UGC in AIO      : {n_aio_ugc}/{n_aio_rows} ({n_aio_ugc / n_aio_rows:.0%})"
    if n_aio_rows
    else "No AIO data"
)
print(f"\nMean UGC share — organic : {df['ugc_share_organic'].mean():.1%}")
print(f"Mean UGC share — AIO     : {df['ugc_share_aio'].mean():.1%}")

In [ ]:
# ── 6a. Overall: mean UGC share — organic vs AIO ──────────────────────────────
org_mean = df["ugc_share_organic"].mean()
aio_mean = df["ugc_share_aio"].mean()

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ["Organic top-10", "AIO sources"],
    [org_mean, aio_mean],
    color=["#3498db", "#e67e22"],
    width=0.5,
)
ax.bar_label(bars, labels=[f"{org_mean:.1%}", f"{aio_mean:.1%}"], padding=5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylim(0, max(org_mean, aio_mean) * 1.5)
ax.set_ylabel("Mean UGC domain share")
ax.set_title("UGC platform share: AIO sources vs Organic top-10")
plt.tight_layout()
plt.show()

In [ ]:
# ── 6b. Per topic: grouped bar ────────────────────────────────────────────────
topic_org = df.groupby("topic")["ugc_share_organic"].mean()
topic_aio = df.groupby("topic")["ugc_share_aio"].mean()
topic_ugc = pd.DataFrame({"Organic": topic_org, "AIO": topic_aio}).sort_index()

x = range(len(topic_ugc))
w = 0.35
fig, ax = plt.subplots(figsize=(max(7, len(topic_ugc) * 1.6), 5))
ax.bar(
    [i - w / 2 for i in x], topic_ugc["Organic"], w, label="Organic", color="#3498db"
)
ax.bar(
    [i + w / 2 for i in x], topic_ugc["AIO"].fillna(0), w, label="AIO", color="#e67e22"
)
ax.set_xticks(list(x))
ax.set_xticklabels(topic_ugc.index, rotation=30, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel("Mean UGC domain share")
ax.set_title("UGC share per topic: AIO vs Organic")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 6c. Per stance: grouped bar ───────────────────────────────────────────────
df_st = df[df["stance"].notna()]

if df_st.empty:
    print("No stance data yet.")
else:
    order = ["pro", "neutrale", "contro"]
    stance_org = df_st.groupby("stance")["ugc_share_organic"].mean().reindex(order)
    stance_aio = df_st.groupby("stance")["ugc_share_aio"].mean().reindex(order)

    x = range(len(order))
    w = 0.35
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        [i - w / 2 for i in x],
        stance_org.fillna(0),
        w,
        label="Organic",
        color="#3498db",
    )
    ax.bar(
        [i + w / 2 for i in x], stance_aio.fillna(0), w, label="AIO", color="#e67e22"
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(order)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.set_ylabel("Mean UGC domain share")
    ax.set_title("UGC share by stance: AIO vs Organic")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print("\nPer-stance UGC share summary:")
    print(pd.DataFrame({"Organic": stance_org, "AIO": stance_aio}).round(3).to_string())

In [ ]:
# ── 6d. Which UGC platforms appear, and where? ────────────────────────────────
def count_platforms(domains_series):
    counts = {u: 0 for u in sorted(UGC_DOMAINS)}
    for dj in domains_series.dropna():
        try:
            domains = json.loads(dj) if isinstance(dj, str) else dj
        except Exception:
            continue
        for d in domains:
            for u in UGC_DOMAINS:
                if d == u or d.endswith("." + u):
                    counts[u] += 1
                    break
    return pd.Series(counts)


n_total = len(df)
n_with_aio = int(df["has_ai_overview"].sum())

org_counts = count_platforms(df["organic_domains"])
aio_counts = count_platforms(df.loc[df["has_ai_overview"], "aio_domains"])

platform_df = pd.DataFrame(
    {
        "Organic (per 100 queries)": org_counts / n_total * 100,
        "AIO (per 100 AIO queries)": aio_counts / n_with_aio * 100 if n_with_aio else 0,
    }
)
platform_df = platform_df[(platform_df > 0).any(axis=1)].sort_values(
    "Organic (per 100 queries)", ascending=False
)

if platform_df.empty:
    print("No UGC platforms found in the current data.")
else:
    x = range(len(platform_df))
    w = 0.35
    fig, ax = plt.subplots(figsize=(max(7, len(platform_df) * 1.3), 5))
    ax.bar(
        [i - w / 2 for i in x],
        platform_df["Organic (per 100 queries)"],
        w,
        label="Organic",
        color="#3498db",
    )
    ax.bar(
        [i + w / 2 for i in x],
        platform_df["AIO (per 100 AIO queries)"],
        w,
        label="AIO",
        color="#e67e22",
    )
    ax.set_xticks(list(x))
    ax.set_xticklabels(platform_df.index, rotation=30, ha="right")
    ax.set_ylabel("Appearances per 100 queries")
    ax.set_title("UGC platform frequency: AIO vs Organic")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(
        f"Normalised by: organic → {n_total} queries | AIO → {n_with_aio} queries with AIO"
    )
    display(platform_df.round(2))

### 6e. YouTube deep-dive: channel vs video citations

YouTube URLs encode the source type in their path:
- `/@handle` / `/user/name` / `/c/name` → **named channel** (identifiable)
- `/channel/UCxxxxxx` → opaque channel ID
- `/watch?v=xxx` / `/shorts/xxx` → **video only** (channel unknown from URL; check title)

For video-only URLs the `title` field from AIO sources often contains the channel name.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from config import YOUTUBE_API_KEY
import re
import requests as _req
from urllib.parse import urlparse, parse_qs

if not YOUTUBE_API_KEY:
    print("⚠️  YOUTUBE_API_KEY not found in .env")
else:
    print("✅ YouTube API key loaded")


# ── URL parser ────────────────────────────────────────────────────────────────
def parse_youtube_url(url):
    """Classify a YouTube URL; return yt_type, yt_name, video_id, channel_id_raw."""
    empty = {
        "yt_type": "invalid",
        "yt_name": None,
        "video_id": None,
        "channel_id_raw": None,
    }
    try:
        parsed = urlparse(url)
        path = parsed.path.rstrip("/")
    except Exception:
        return empty

    base = {"video_id": None, "channel_id_raw": None, "yt_name": None}

    for pattern, typ in [
        (r"^/@([^/]+)", "handle"),
        (r"^/user/([^/]+)", "username"),
        (r"^/c/([^/]+)", "custom"),
    ]:
        m = re.match(pattern, path)
        if m:
            return {**base, "yt_type": typ, "yt_name": m.group(1)}

    m = re.match(r"^/channel/(UC[A-Za-z0-9_-]+)", path)
    if m:
        return {**base, "yt_type": "channel_id", "channel_id_raw": m.group(1)}

    m = re.match(r"^/shorts/([A-Za-z0-9_-]+)", path)
    if m:
        return {**base, "yt_type": "short", "video_id": m.group(1)}

    qs = parse_qs(parsed.query)
    if "v" in qs:
        return {**base, "yt_type": "video", "video_id": qs["v"][0]}
    if "/watch" in path:
        return {**base, "yt_type": "video"}

    return {**base, "yt_type": "other", "yt_name": path}


# ── YouTube Data API v3 helpers ───────────────────────────────────────────────
def _yt_batch(endpoint, id_param, ids, api_key):
    """Batch YouTube API call (max 50 IDs per request). Returns {id: snippet}."""
    results = {}
    for i in range(0, len(ids), 50):
        batch = [x for x in ids[i : i + 50] if x]
        if not batch:
            continue
        try:
            resp = _req.get(
                f"https://www.googleapis.com/youtube/v3/{endpoint}",
                params={"part": "snippet", id_param: ",".join(batch), "key": api_key},
                timeout=15,
            )
            resp.raise_for_status()
            for item in resp.json().get("items", []):
                results[item["id"]] = item.get("snippet", {})
        except Exception as e:
            print(f"  YouTube API error ({endpoint}): {e}")
    return results


# ── Build raw df_yt ───────────────────────────────────────────────────────────
yt_rows = []
for _, row in df[df["has_ai_overview"]].iterrows():
    aio_raw = row.get("aio_sources")
    if not aio_raw or isinstance(aio_raw, float):
        continue
    try:
        sources = json.loads(aio_raw) if isinstance(aio_raw, str) else aio_raw
    except Exception:
        continue
    for src in sources:
        link = src.get("link", "")
        title = src.get("title", "")
        if "youtube.com" not in link and "youtu.be" not in link:
            continue
        yt_rows.append(
            {
                "query": row["query"],
                "topic": row["topic"],
                "stance": row.get("stance"),
                "title": title,
                "link": link,
                **parse_youtube_url(link),
            }
        )

df_yt = pd.DataFrame(yt_rows)
print(f"\nYouTube citations found: {len(df_yt)}")
if df_yt.empty:
    print("No YouTube citations in the current data.")
else:
    print("URL type breakdown (before API):")
    print(df_yt["yt_type"].value_counts().to_string())


# ── YouTube API enrichment ────────────────────────────────────────────────────
if not df_yt.empty and YOUTUBE_API_KEY:
    video_ids = df_yt.loc[df_yt["video_id"].notna(), "video_id"].unique().tolist()
    chan_ids = (
        df_yt.loc[df_yt["channel_id_raw"].notna(), "channel_id_raw"].unique().tolist()
    )

    vid_snippets = {}
    if video_ids:
        print(f"  → Fetching {len(video_ids)} video(s) from YouTube API…")
        vid_snippets = _yt_batch("videos", "id", video_ids, YOUTUBE_API_KEY)
        print(f"    resolved {len(vid_snippets)}/{len(video_ids)}")

    chan_snippets = {}
    if chan_ids:
        print(f"  → Fetching {len(chan_ids)} channel(s) from YouTube API…")
        chan_snippets = _yt_batch("channels", "id", chan_ids, YOUTUBE_API_KEY)
        print(f"    resolved {len(chan_snippets)}/{len(chan_ids)}")

    def _enrich(row):
        if row["video_id"] and row["video_id"] in vid_snippets:
            s = vid_snippets[row["video_id"]]
            return pd.Series(
                {
                    "channel_title": s.get("channelTitle"),
                    "channel_id": s.get("channelId"),
                    "api_resolved": True,
                }
            )
        if row["channel_id_raw"] and row["channel_id_raw"] in chan_snippets:
            s = chan_snippets[row["channel_id_raw"]]
            return pd.Series(
                {
                    "channel_title": s.get("title"),
                    "channel_id": row["channel_id_raw"],
                    "api_resolved": True,
                }
            )
        return pd.Series(
            {
                "channel_title": row["yt_name"],
                "channel_id": row.get("channel_id_raw"),
                "api_resolved": False,
            }
        )

    df_yt[["channel_title", "channel_id", "api_resolved"]] = df_yt.apply(
        _enrich, axis=1
    )
else:
    df_yt["channel_title"] = df_yt.get("yt_name")
    df_yt["channel_id"] = df_yt.get("channel_id_raw")
    df_yt["api_resolved"] = False

df_yt["channel_label"] = df_yt["channel_title"].fillna("(unknown)")

resolved = df_yt["api_resolved"].sum()
print(f"\nAPI-resolved channel info: {resolved}/{len(df_yt)} citations")
print(
    f"Channel label known      : {(df_yt['channel_label'] != '(unknown)').sum()}/{len(df_yt)}"
)


# ── URL-type pie ──────────────────────────────────────────────────────────────
if not df_yt.empty:
    type_counts = df_yt["yt_type"].value_counts()
    TYPE_COLORS = {
        "handle": "#27ae60",
        "username": "#2ecc71",
        "custom": "#82e0aa",
        "channel_id": "#f39c12",
        "video": "#e74c3c",
        "short": "#c0392b",
        "other": "#95a5a6",
        "invalid": "#7f8c8d",
    }
    colors = [TYPE_COLORS.get(t, "#bdc3c7") for t in type_counts.index]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(
        type_counts.values,
        labels=type_counts.index,
        colors=colors,
        autopct="%1.0f%%",
        startangle=140,
        wedgeprops={"linewidth": 1, "edgecolor": "white"},
    )
    ax.set_title("YouTube citations in AIO — URL type")
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Top channels cited (URL name + title-derived) ─────────────────────────────
if not df_yt.empty:
    top_channels = (
        df_yt[df_yt["channel_label"] != "(unknown)"]["channel_label"]
        .value_counts()
        .head(20)
    )

    if not top_channels.empty:
        fig, ax = plt.subplots(figsize=(10, max(4, len(top_channels) * 0.42)))
        ax.barh(
            top_channels.index[::-1],
            top_channels.values[::-1],
            color="#e74c3c",
            alpha=0.85,
            edgecolor="white",
        )
        for i, v in enumerate(top_channels.values[::-1]):
            ax.text(v + 0.05, i, str(v), va="center", fontsize=9)
        ax.set_xlabel("Citations in AIO")
        ax.set_title("Top YouTube channels cited by Google AIO")
        plt.tight_layout()
        plt.show()
    else:
        print("No named channels found — all citations are video-only URLs.")

    # ── Full table ────────────────────────────────────────────────────────────
    print("\nAll YouTube citations:")
    display(
        df_yt[["topic", "stance", "yt_type", "channel_label", "title", "link"]]
        .sort_values(["yt_type", "channel_label"])
        .reset_index(drop=True)
        .style.set_properties(**{"text-align": "left"})
    )

## 7. Top Cited Domains: AIO vs Organic

Absolute citation counts for every domain across all records.  
Each row in `organic_domains` / `aio_domains` is a JSON list; we explode them and count appearances.  
(Repeated queries count multiple times — we also show a deduplicated view.)

In [ ]:
def explode_domains(series, label):
    """Return a flat Series of domain strings from a column of JSON lists."""
    rows = []
    for dj in series.dropna():
        try:
            domains = json.loads(dj) if isinstance(dj, str) else dj
            rows.extend(d for d in domains if d)
        except Exception:
            pass
    return pd.Series(rows, name=label)


TOP_N = 15

# ── Raw counts (all records, including repeated queries) ──────────────────────
aio_domains_flat = explode_domains(df.loc[df["has_ai_overview"], "aio_domains"], "aio")
org_domains_flat = explode_domains(df["organic_domains"], "organic")

top_aio = aio_domains_flat.value_counts().head(TOP_N)
top_org = org_domains_flat.value_counts().head(TOP_N)

print(f"Unique domains in AIO     : {aio_domains_flat.nunique()}")
print(f"Unique domains in Organic : {org_domains_flat.nunique()}")
print(f"Total AIO citations       : {len(aio_domains_flat)}")
print(f"Total Organic citations   : {len(org_domains_flat)}")

fig, (ax_aio, ax_org) = plt.subplots(1, 2, figsize=(14, 6))

# AIO panel
ax_aio.barh(top_aio.index[::-1], top_aio.values[::-1], color="#e67e22")
ax_aio.set_xlabel("Appearances (raw)")
ax_aio.set_title(f"Top {TOP_N} domains — AIO sources")
for i, v in enumerate(top_aio.values[::-1]):
    ax_aio.text(v + 0.3, i, str(v), va="center", fontsize=8)

# Organic panel
ax_org.barh(top_org.index[::-1], top_org.values[::-1], color="#3498db")
ax_org.set_xlabel("Appearances (raw)")
ax_org.set_title(f"Top {TOP_N} domains — Organic top-10")
for i, v in enumerate(top_org.values[::-1]):
    ax_org.text(v + 0.3, i, str(v), va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 7b. Deduplicated: each (query, domain) pair counted once ─────────────────
def explode_domains_dedup(df_in, domains_col, query_col="query"):
    rows = []
    for _, row in df_in[[query_col, domains_col]].dropna().iterrows():
        try:
            domains = (
                json.loads(row[domains_col])
                if isinstance(row[domains_col], str)
                else row[domains_col]
            )
            for d in set(domains):
                if d:
                    rows.append(d)
        except Exception:
            pass
    return pd.Series(rows)


top_aio_dd = (
    explode_domains_dedup(df[df["has_ai_overview"]], "aio_domains")
    .value_counts()
    .head(TOP_N)
)
top_org_dd = explode_domains_dedup(df, "organic_domains").value_counts().head(TOP_N)

fig, (ax_aio, ax_org) = plt.subplots(1, 2, figsize=(14, 6))

ax_aio.barh(top_aio_dd.index[::-1], top_aio_dd.values[::-1], color="#e67e22")
ax_aio.set_xlabel("Unique queries citing domain")
ax_aio.set_title(f"Top {TOP_N} domains — AIO (deduplicated)")
for i, v in enumerate(top_aio_dd.values[::-1]):
    ax_aio.text(v + 0.1, i, str(v), va="center", fontsize=8)

ax_org.barh(top_org_dd.index[::-1], top_org_dd.values[::-1], color="#3498db")
ax_org.set_xlabel("Unique queries citing domain")
ax_org.set_title(f"Top {TOP_N} domains — Organic (deduplicated)")
for i, v in enumerate(top_org_dd.values[::-1]):
    ax_org.text(v + 0.1, i, str(v), va="center", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── 7c. AIO rank vs Organic rank for shared domains ──────────────────────────
aio_ranks = (
    aio_domains_flat.value_counts()
    .rank(method="min", ascending=False)
    .rename("aio_rank")
)
org_ranks = (
    org_domains_flat.value_counts()
    .rank(method="min", ascending=False)
    .rename("org_rank")
)

shared = pd.concat([aio_ranks, org_ranks], axis=1).dropna()
shared["aio_count"] = aio_domains_flat.value_counts()
shared["org_count"] = org_domains_flat.value_counts()
shared = shared.sort_values("aio_count", ascending=False)

print(f"Domains appearing in both AIO and Organic: {len(shared)}")
display(shared.head(20).astype({"aio_rank": int, "org_rank": int}))

if len(shared) >= 3:
    fig, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(
        shared["org_rank"],
        shared["aio_rank"],
        s=shared["aio_count"] * 10,
        alpha=0.7,
        color="#9b59b6",
        edgecolors="white",
    )
    for domain, row in shared.head(12).iterrows():
        ax.annotate(
            domain,
            (row["org_rank"], row["aio_rank"]),
            fontsize=7,
            xytext=(4, 4),
            textcoords="offset points",
        )
    ax.set_xlabel("Organic rank (1 = most cited)")
    ax.set_ylabel("AIO rank (1 = most cited)")
    ax.set_title("Domain rank: AIO vs Organic\n(bubble size = AIO citation count)")
    ax.invert_xaxis()
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

In [ ]:
# ── 7d. Heatmap: domain citation rate per topic (% of queries) ────────────────
TOP_DOMAINS_HEAT = 20


def domain_topic_pct(df_in, domains_col, topic_col="topic"):
    """
    Returns a (domain × topic) DataFrame where each cell is the % of queries
    in that topic that cite the domain (deduplicated: one count per query).
    """
    n_per_topic = df_in.groupby(topic_col).size()
    rows = []
    for _, row in df_in[[topic_col, domains_col]].dropna().iterrows():
        try:
            doms = (
                json.loads(row[domains_col])
                if isinstance(row[domains_col], str)
                else row[domains_col]
            )
            for d in set(doms):
                if d:
                    rows.append({"topic": row[topic_col], "domain": d})
        except Exception:
            pass
    if not rows:
        return pd.DataFrame()
    flat = pd.DataFrame(rows)
    mat = (
        flat.groupby(["topic", "domain"]).size().unstack(fill_value=0)
    )  # topic × domain
    return (mat.div(n_per_topic, axis=0) * 100).T  # → domain × topic


aio_mat = domain_topic_pct(df[df["has_ai_overview"]], "aio_domains")
org_mat = domain_topic_pct(df, "organic_domains")

# Select top domains by their peak % across any topic
top_aio_doms = (
    aio_mat.max(axis=1).nlargest(TOP_DOMAINS_HEAT).index if not aio_mat.empty else []
)
top_org_doms = (
    org_mat.max(axis=1).nlargest(TOP_DOMAINS_HEAT).index if not org_mat.empty else []
)

n_topics = max(
    len(aio_mat.columns) if not aio_mat.empty else 1,
    len(org_mat.columns) if not org_mat.empty else 1,
)
fig_w = max(6, n_topics * 1.4)

# ── AIO heatmap ───────────────────────────────────────────────────────────────
if not aio_mat.empty and len(top_aio_doms):
    fig, ax = plt.subplots(figsize=(fig_w, max(4, len(top_aio_doms) * 0.45)))
    sns.heatmap(
        aio_mat.loc[top_aio_doms].round(0).astype(int),
        annot=True,
        fmt="d",
        cmap="YlOrRd",
        vmin=0,
        vmax=100,
        linewidths=0.4,
        linecolor="#eee",
        cbar_kws={"label": "% of AIO queries citing domain"},
        ax=ax,
    )
    ax.set_title("Domain citation rate per topic — AIO sources (%)")
    ax.set_xlabel("Topic")
    ax.set_ylabel("Domain")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

# ── Organic heatmap ───────────────────────────────────────────────────────────
if not org_mat.empty and len(top_org_doms):
    fig, ax = plt.subplots(figsize=(fig_w, max(4, len(top_org_doms) * 0.45)))
    sns.heatmap(
        org_mat.loc[top_org_doms].round(0).astype(int),
        annot=True,
        fmt="d",
        cmap="Blues",
        vmin=0,
        vmax=100,
        linewidths=0.4,
        linecolor="#eee",
        cbar_kws={"label": "% of queries citing domain"},
        ax=ax,
    )
    ax.set_title("Domain citation rate per topic — Organic top-10 (%)")
    ax.set_xlabel("Topic")
    ax.set_ylabel("Domain")
    ax.tick_params(axis="x", rotation=30)
    ax.tick_params(axis="y", rotation=0)
    plt.tight_layout()
    plt.show()

## 8. AIO Content — General Stats

Focused only on records where Google returned an AI Overview.  
Metrics: number of cited sources, text length (characters & words), breakdowns by topic and stance.

In [ ]:
# ── 8. Setup: AIO-only slice + derived text metrics ──────────────────────────
df_a = df[df["has_ai_overview"]].copy()
df_a["aio_source_count"] = (
    pd.to_numeric(df_a["aio_source_count"], errors="coerce").fillna(0).astype(int)
)
df_a["aio_text_chars"] = df_a["aio_text"].str.len()
df_a["aio_text_words"] = df_a["aio_text"].str.split().str.len()

n_with_text = df_a["aio_text"].notna().sum()
n_no_text = df_a["aio_text"].isna().sum()

print(
    f"AIO records          : {len(df_a)} / {len(df)} total ({len(df_a) / len(df):.0%})"
)
print(f"  with text content  : {n_with_text}")
print(f"  empty AIO (no text): {n_no_text}")
print()

summary = (
    df_a[["aio_source_count", "aio_text_chars", "aio_text_words"]]
    .describe()
    .loc[["count", "mean", "std", "min", "25%", "50%", "75%", "max"]]
    .rename(
        columns={
            "aio_source_count": "sources cited",
            "aio_text_chars": "text length (chars)",
            "aio_text_words": "text length (words)",
        }
    )
    .round(1)
)
display(summary)

In [ ]:
# ── 8a. Distributions: source count & text word count ────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sources per AIO
ax = axes[0]
ax.hist(
    df_a["aio_source_count"],
    bins=range(0, df_a["aio_source_count"].max() + 2),
    color="#e67e22",
    edgecolor="white",
    align="left",
)
ax.axvline(
    df_a["aio_source_count"].mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_a['aio_source_count'].mean():.1f}",
)
ax.axvline(
    df_a["aio_source_count"].median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_a['aio_source_count'].median():.0f}",
)
ax.set_xlabel("Sources cited per AIO")
ax.set_ylabel("Number of queries")
ax.set_title("Source count distribution")
ax.legend(fontsize=8)

# Text length — characters
ax = axes[1]
df_text = df_a["aio_text_chars"].dropna()
ax.hist(df_text, bins=20, color="#9b59b6", edgecolor="white")
ax.axvline(
    df_text.mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_text.mean():.0f}",
)
ax.axvline(
    df_text.median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_text.median():.0f}",
)
ax.set_xlabel("AIO text length (characters)")
ax.set_ylabel("Number of queries")
ax.set_title("Text length — characters")
ax.legend(fontsize=8)

# Text length — words
ax = axes[2]
df_words = df_a["aio_text_words"].dropna()
ax.hist(df_words, bins=20, color="#1abc9c", edgecolor="white")
ax.axvline(
    df_words.mean(),
    color="black",
    linestyle="--",
    linewidth=1.2,
    label=f"mean {df_words.mean():.0f}",
)
ax.axvline(
    df_words.median(),
    color="grey",
    linestyle=":",
    linewidth=1.2,
    label=f"median {df_words.median():.0f}",
)
ax.set_xlabel("AIO text length (words)")
ax.set_ylabel("Number of queries")
ax.set_title("Text length — words")
ax.legend(fontsize=8)

plt.suptitle("AIO content distributions", y=1.02, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 8b. Source count per topic ────────────────────────────────────────────────
topics_order = sorted(df_a["topic"].dropna().unique())

fig, axes = plt.subplots(1, 2, figsize=(max(10, len(topics_order) * 2.2), 5))

# Sources
ax = axes[0]
sns.boxplot(
    data=df_a,
    x="topic",
    y="aio_source_count",
    order=topics_order,
    color="#e67e22",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_a,
    x="topic",
    y="aio_source_count",
    order=topics_order,
    color="black",
    size=3,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics_order, rotation=30, ha="right")
ax.set_ylabel("Sources cited")
ax.set_xlabel("")
ax.set_title("Sources cited per AIO — by topic")

# Word count
ax = axes[1]
df_a_text = df_a[df_a["aio_text_words"].notna()]
sns.boxplot(
    data=df_a_text,
    x="topic",
    y="aio_text_words",
    order=topics_order,
    color="#1abc9c",
    width=0.5,
    ax=ax,
)
sns.stripplot(
    data=df_a_text,
    x="topic",
    y="aio_text_words",
    order=topics_order,
    color="black",
    size=3,
    alpha=0.5,
    jitter=True,
    ax=ax,
)
ax.set_xticklabels(topics_order, rotation=30, ha="right")
ax.set_ylabel("Word count")
ax.set_xlabel("")
ax.set_title("AIO text length (words) — by topic")

plt.tight_layout()
plt.show()

print("\nMean sources cited per topic:")
print(df_a.groupby("topic")["aio_source_count"].describe().round(1).to_string())

In [ ]:
# ── 8c. Source count & text length per stance ─────────────────────────────────
df_a_st = df_a[df_a["stance"].notna()]

if df_a_st.empty:
    print("No stance data yet.")
else:
    stance_order = ["pro", "neutrale", "contro"]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    ax = axes[0]
    sns.boxplot(
        data=df_a_st,
        x="stance",
        y="aio_source_count",
        order=stance_order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_st,
        x="stance",
        y="aio_source_count",
        order=stance_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Query stance")
    ax.set_ylabel("Sources cited")
    ax.set_title("Sources cited per AIO — by stance")

    ax = axes[1]
    df_a_st_text = df_a_st[df_a_st["aio_text_words"].notna()]
    sns.boxplot(
        data=df_a_st_text,
        x="stance",
        y="aio_text_words",
        order=stance_order,
        palette=PALETTE,
        width=0.5,
        ax=ax,
    )
    sns.stripplot(
        data=df_a_st_text,
        x="stance",
        y="aio_text_words",
        order=stance_order,
        color="black",
        size=4,
        alpha=0.5,
        jitter=True,
        ax=ax,
    )
    ax.set_xlabel("Query stance")
    ax.set_ylabel("Word count")
    ax.set_title("AIO text length (words) — by stance")

    plt.tight_layout()
    plt.show()

    print("\nSources cited — by stance:")
    print(
        df_a_st.groupby("stance")["aio_source_count"]
        .describe()
        .round(1)
        .reindex(stance_order)
        .to_string()
    )
    print("\nText word count — by stance:")
    print(
        df_a_st.groupby("stance")["aio_text_words"]
        .describe()
        .round(1)
        .reindex(stance_order)
        .to_string()
    )